# Chapter 17：Attention Memory Analysis

本章分析普通 attention 与 FlashAttention 的中间存储差异。Q/K/V 和输出是 `[B,H,S,D]`，而 scores/probs 是 `[B,H,S,S]`。

In [ ]:
import math
import torch

## 普通 attention 的 activation memory

主要中间 tensor 是 scores 和 probs；某些实现还可能有 masked scores 和训练时 dropout mask。由于它们包含 SxS，S 翻倍时内存约变为四倍。

In [ ]:
def format_bytes(num_bytes: float) -> str:
    units = ("B", "KiB", "MiB", "GiB", "TiB")
    value = float(num_bytes)
    for unit in units:
        if abs(value) < 1024 or unit == units[-1]:
            return f"{value:.2f} {unit}"
        value /= 1024
    raise AssertionError("unreachable")


def estimate_attention_tensors(B: int, H: int, S: int, D: int, dtype_bytes: int = 2) -> dict[str, float]:
    qkv = 3 * B * H * S * D * dtype_bytes
    scores = B * H * S * S * dtype_bytes
    probabilities = scores
    output = B * H * S * D * dtype_bytes
    total = qkv + scores + probabilities + output
    return {
        "qkv_memory": qkv,
        "scores_memory": scores,
        "probs_memory": probabilities,
        "output_memory": output,
        "total_important_activation_memory": total,
        "scores_probs_to_qkv_ratio": (scores + probabilities) / qkv,
    }


def print_attention_memory_table() -> None:
    B, H, D = 1, 16, 64
    print(f"Theory table: B={B}, H={H}, D={D}, fp16")
    print(f"{'S':>6} {'QKV':>12} {'scores':>12} {'probs':>12} {'total':>12} {'S/P : QKV':>12}")
    print("-" * 72)
    for S in (128, 256, 512, 1024, 2048, 4096):
        item = estimate_attention_tensors(B, H, S, D)
        print(f"{S:6d} {format_bytes(item['qkv_memory']):>12} {format_bytes(item['scores_memory']):>12} {format_bytes(item['probs_memory']):>12} {format_bytes(item['total_important_activation_memory']):>12} {item['scores_probs_to_qkv_ratio']:12.2f}")

    working_set = estimate_flashattention_working_set(32, 32, D)
    print("\nTeaching FlashAttention tile working-set estimate (BLOCK_M=32, BLOCK_N=32):")
    for name, value in working_set.items():
        print(f"  {name:<20} {format_bytes(value)}")
    print("This is an educational estimate, not exact register or shared-memory usage.")

## FlashAttention tile working set

FlashAttention 不把完整 scores/probs 写入 HBM，而是分块加载 Q/K/V，在 tile 内用 online softmax 的 m/l/acc 累计结果。计算复杂度仍约为 `O(S^2*D)`；改善的是中间存储和 HBM 流量。

下面只估算 Q/K/V block、score block、fp32 acc 和 m/l。这不是精确的寄存器或 shared-memory 报告。

In [ ]:
def estimate_flashattention_working_set(BLOCK_M: int, BLOCK_N: int, D: int, dtype_bytes: int = 2, acc_bytes: int = 4) -> dict[str, float]:
    q_block = BLOCK_M * D * dtype_bytes
    k_block = BLOCK_N * D * dtype_bytes
    v_block = BLOCK_N * D * dtype_bytes
    score_block = BLOCK_M * BLOCK_N * acc_bytes
    accumulator = BLOCK_M * D * acc_bytes
    ml_state = 2 * BLOCK_M * acc_bytes
    return {
        "q_block": q_block,
        "k_block": k_block,
        "v_block": v_block,
        "score_block": score_block,
        "accumulator": accumulator,
        "m_l_state": ml_state,
        "total_working_set": q_block + k_block + v_block + score_block + accumulator + ml_state,
    }

In [ ]:
print_attention_memory_table()

## 显式 materialized attention

为了观察普通 attention，本函数显式创建 scores 和 probs，不使用可能隐式融合的 optimized SDPA。causal mask 保留 `key_index <= query_index`。

In [ ]:
def naive_attention_materialized(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, causal: bool = False) -> torch.Tensor:
    if q.ndim != 4 or q.shape != k.shape or q.shape != v.shape:
        raise ValueError("q, k, and v must share shape [B, H, S, D]")
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(q.shape[-1])
    if causal:
        positions = torch.arange(q.shape[-2], device=q.device)
        scores = scores.masked_fill(positions[None, :] > positions[:, None], -float("inf"))
    probabilities = torch.softmax(scores, dim=-1)
    return torch.matmul(probabilities, v)

## CUDA peak-memory 观测

`max_memory_allocated` 会受 PyTorch allocator、cache、临时 workspace 和具体实现影响，不等于理论估算。实际 demo 只运行到 S=1024，避免 OOM。

本章不依赖 Chapter 13；可选 Triton 比较未启用时会明确跳过。

In [ ]:
def measure_peak_memory(fn, *args, **kwargs) -> tuple[torch.Tensor, int]:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for peak-memory measurement")
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    baseline = torch.cuda.memory_allocated()
    output = fn(*args, **kwargs)
    torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated()
    return output, peak - baseline


def run_peak_memory_demo() -> None:
    if not torch.cuda.is_available():
        print("Skipping actual peak-memory demo because CUDA is not available.")
        print("Skipping Triton FlashAttention comparison because chapter_13 implementation is unavailable.")
        return
    print("\nMaterialized attention peak-memory demo (allocator/cache effects may differ from theory):")
    for S in (128, 256, 512, 1024):
        q = torch.randn(1, 8, S, 64, device="cuda", dtype=torch.float16)
        k = torch.randn_like(q)
        v = torch.randn_like(q)
        output, peak = measure_peak_memory(naive_attention_materialized, q, k, v)
        print(f"S={S:4d}: peak additional allocated memory = {format_bytes(peak)}")
        del output, q, k, v
    print("Skipping Triton FlashAttention comparison because chapter_13 implementation is unavailable.")

In [ ]:
run_peak_memory_demo()

## 小结

普通 attention 的 scores/probs 随 S² 增长。FlashAttention 没有减少总 attention 计算量，而是避免完整 SxS 中间矩阵的 HBM 存储和反复读写。

**练习**：将 H 从 8 改为 16，比较 scores/probs 和 QKV memory 的变化。